# Module 01, Notebook 3: Seasonal Adjustments in ITS

## Learning Objectives
By completing this notebook, you will:
1. Understand when and why seasonal adjustment is necessary in ITS
2. Implement month dummy variable seasonal controls
3. Implement Fourier (sine/cosine) seasonal controls
4. Compare seasonal adjustment methods using LOO cross-validation
5. Visualize and interpret seasonally-adjusted causal estimates

## The Seasonal Problem in ITS

Many time series have strong seasonal patterns. If an intervention occurs at a seasonal peak or trough, the naive ITS estimate confounds the intervention effect with the seasonal variation:

```
Observed change at t* = True intervention effect + Seasonal change
```

In this notebook, we work with a dataset where the intervention coincides with a seasonal transition. We compare unadjusted, month-dummy-adjusted, and Fourier-adjusted models.

## Estimated Time: 15 minutes

---

In [ ]:
learning_objectives(["Understand when and why seasonal adjustment is necessary in ITS", "Implement month dummy variable seasonal controls", "Implement Fourier (sine/cosine) seasonal controls", "Compare seasonal adjustment methods using LOO cross-validation", "Visualize and interpret seasonally-adjusted causal estimates"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import causalpy as cp
import arviz as az
from scipy import stats

np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline

## 1. Generate a Dataset with Strong Seasonality

We simulate monthly flu-related hospital visits. Flu visits are strongly seasonal (peak in winter), and we analyze a vaccination campaign that launched in September (just before flu season).

In [ ]:
# Dataset: Monthly flu-related emergency department visits per 10,000
# Intervention: Universal flu vaccination campaign (free vouchers, incentives)
# Intervention launch: September (month 20, 0-indexed)
# The seasonal peak in flu visits is November-February

np.random.seed(2024)
n_months = 48
intervention_month = 20  # September of year 2

months = np.arange(n_months)
calendar_month = months % 12  # 0=Jan, 8=Sep, 11=Dec

# Strong seasonal pattern: peak in winter (months 11, 0, 1, 2)
# Amplitude of 40 visits/10k (very pronounced seasonality)
seasonal_amplitude = 40.0
seasonal_effect = (
    seasonal_amplitude * np.cos(2 * np.pi * calendar_month / 12)
    + (seasonal_amplitude * 0.3) * np.cos(4 * np.pi * calendar_month / 12)  # 2nd harmonic
)

# Treatment indicators
treated = (months >= intervention_month).astype(float)
t_post = np.maximum(months - intervention_month, 0).astype(float)

# True intervention effect: -8 visits/10k immediate, -0.2/month additional decline
true_level_change = -8.0
true_slope_change = -0.2

flu_visits = (
    80.0                            # Baseline
    + 0.1 * months                  # Slight upward secular trend
    + seasonal_effect               # Strong winter peak
    + true_level_change * treated   # Vaccination campaign effect
    + true_slope_change * t_post * treated
    + np.random.normal(0, 5.0, n_months)  # Noise
)

# Date range
dates = pd.date_range(start="2019-01", periods=n_months, freq="MS")
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

df = pd.DataFrame({
    "date": dates,
    "month": months,
    "flu_visits": flu_visits,
    "treated": treated,
    "t_post": t_post,
    "calendar_month": calendar_month,
    "month_name": [month_names[m] for m in calendar_month],
})

print(f"Intervention date: month {intervention_month} = {dates[intervention_month].strftime('%B %Y')}")
print(f"Note: Intervention in September, just BEFORE the flu season peak!")
print(f"This means the raw series will rise after the intervention due to seasonality.")

In [ ]:
# Visualize the seasonal challenge

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# Raw series
ax = axes[0]
intervention_date = dates[intervention_month]

pre_mask = df["treated"] == 0
post_mask = df["treated"] == 1

ax.plot(df.loc[pre_mask, "date"], df.loc[pre_mask, "flu_visits"],
        "o-", color="#2c3e50", linewidth=1.5, markersize=4, label="Pre-intervention")
ax.plot(df.loc[post_mask, "date"], df.loc[post_mask, "flu_visits"],
        "s-", color="#e74c3c", linewidth=1.5, markersize=4, label="Post-intervention")
ax.axvline(intervention_date, color="#27ae60", linestyle="--", linewidth=2.5,
           label=f"Vaccination campaign launch ({intervention_date.strftime('%b %Y')})")

ax.set_ylabel("Flu ED Visits per 10,000", fontsize=12)
ax.set_title("Flu Visits: Strong Seasonality + Intervention in September", fontsize=13)
ax.legend(fontsize=11)
import matplotlib.dates as mdates
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

# Seasonal pattern
ax2 = axes[1]
monthly_avg = df.groupby("calendar_month")["flu_visits"].mean()
bars = ax2.bar(range(12), monthly_avg.values, color="#3498db", edgecolor="white", alpha=0.85)
ax2.set_xticks(range(12))
ax2.set_xticklabels(month_names)
ax2.axvline(8 - 0.5, color="#27ae60", linestyle="--", linewidth=2, label="Intervention (Sep)")
ax2.set_ylabel("Average Flu Visits per 10,000", fontsize=12)
ax2.set_title("Average Seasonal Pattern (All Months)", fontsize=13)
ax2.legend()

plt.tight_layout()
plt.show()

print("\nKey challenge:")
print("  The vaccination campaign launched in September.")
print("  Flu visits NATURALLY RISE from October through February.")
print("  A naive ITS would see a rise after the intervention and wrongly conclude")
print("  the campaign INCREASED flu visits!")

## 2. Naive ITS (No Seasonal Adjustment)

First, show what happens when we ignore seasonality.

In [ ]:
# Naive ITS: No seasonal controls
# Expected: biased estimate — the model attributes the seasonal rise to the intervention

model_naive = cp.pymc_experiments.InterruptedTimeSeries(
    data=df,
    treatment_time=intervention_month,
    formula="flu_visits ~ 1 + month + treated + t_post",
    model=cp.pymc_models.LinearRegression(
        sample_kwargs={"draws": 1000, "tune": 1000, "chains": 4,
                       "progressbar": True, "random_seed": 42}
    ),
)

beta_naive = model_naive.idata.posterior["treated"].values.flatten()
print(f"Naive ITS level change estimate: {beta_naive.mean():+.2f}")
print(f"True level change:               {true_level_change:+.2f}")
print()
print("The naive model may attribute the seasonal rise to the intervention!")

## 3. Month Dummy Seasonal Adjustment

In [ ]:
# Month dummy seasonal adjustment
# C(calendar_month) creates 12 dummy variables (one for each month)
# This is non-parametric: each month gets its own fixed effect

model_month_dummies = cp.pymc_experiments.InterruptedTimeSeries(
    data=df,
    treatment_time=intervention_month,
    formula="flu_visits ~ 1 + month + treated + t_post + C(calendar_month)",
    model=cp.pymc_models.LinearRegression(
        sample_kwargs={"draws": 1000, "tune": 1000, "chains": 4,
                       "progressbar": True, "random_seed": 42}
    ),
)

beta_dummies = model_month_dummies.idata.posterior["treated"].values.flatten()
print(f"Month-dummy ITS level change estimate: {beta_dummies.mean():+.2f}")
print(f"True level change:                     {true_level_change:+.2f}")

## 4. Fourier Seasonal Adjustment

In [ ]:
# Fourier seasonal adjustment
# Adds sine/cosine pairs for smooth seasonal patterns
# Fewer parameters than month dummies, better for short time series

def add_fourier_terms(df, period=12, n_terms=3):
    """Add Fourier sine/cosine pairs to model seasonal patterns."""
    df = df.copy()
    for k in range(1, n_terms + 1):
        df[f"sin_{k}"] = np.sin(2 * np.pi * k * df["month"] / period)
        df[f"cos_{k}"] = np.cos(2 * np.pi * k * df["month"] / period)
    return df

# We use 3 Fourier harmonics (captures fundamental + 2nd and 3rd harmonics)
df_fourier = add_fourier_terms(df, period=12, n_terms=3)

fourier_terms = "sin_1 + cos_1 + sin_2 + cos_2 + sin_3 + cos_3"

model_fourier = cp.pymc_experiments.InterruptedTimeSeries(
    data=df_fourier,
    treatment_time=intervention_month,
    formula=f"flu_visits ~ 1 + month + treated + t_post + {fourier_terms}",
    model=cp.pymc_models.LinearRegression(
        sample_kwargs={"draws": 1000, "tune": 1000, "chains": 4,
                       "progressbar": True, "random_seed": 42}
    ),
)

beta_fourier = model_fourier.idata.posterior["treated"].values.flatten()
print(f"Fourier ITS level change estimate: {beta_fourier.mean():+.2f}")
print(f"True level change:                 {true_level_change:+.2f}")

## 5. Compare All Three Models

In [ ]:
# Side-by-side comparison of level change estimates

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

models_data = [
    (beta_naive, "Naive\n(No Seasonal Adjustment)", "#95a5a6"),
    (beta_dummies, "Month Dummies\n(Non-parametric)", "#3498db"),
    (beta_fourier, "Fourier Terms\n(Smooth Seasonal)", "#27ae60"),
]

for ax, (samples, title, color) in zip(axes, models_data):
    ax.hist(samples, bins=50, edgecolor="white", color=color, alpha=0.85)
    ax.axvline(0, color="black", linestyle="--", linewidth=2, label="No effect")
    ax.axvline(samples.mean(), color="#2c3e50", linewidth=2.5,
               label=f"Est: {samples.mean():.1f}")
    ax.axvline(true_level_change, color="#e74c3c", linewidth=2, linestyle=":",
               label=f"True: {true_level_change:.1f}")
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Level Change (flu visits/10k)", fontsize=10)
    ax.legend(fontsize=9)

axes[0].set_ylabel("Posterior Density", fontsize=11)
plt.suptitle("Seasonal Adjustment Methods: Impact on Level Change Estimate", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("Summary:")
print(f"  True level change:              {true_level_change:+.2f}")
print(f"  Naive estimate:                 {beta_naive.mean():+.2f}")
print(f"  Month-dummy estimate:           {beta_dummies.mean():+.2f}")
print(f"  Fourier estimate:               {beta_fourier.mean():+.2f}")
print()
print("Without seasonal adjustment, the estimate may be biased toward zero or")
print("even positive (false null or reversed sign).")

In [ ]:
# Model comparison using LOO cross-validation

comparison = az.compare(
    {
        "naive": model_naive.idata,
        "month_dummies": model_month_dummies.idata,
        "fourier_3": model_fourier.idata,
    },
    ic="loo",
)

print("LOO Model Comparison:")
print(comparison[["elpd_loo", "p_loo", "elpd_diff", "weight"]].to_string())
print()
print("Higher elpd_loo = better predictive accuracy")
print("elpd_diff = difference from best model")
print("weight = relative evidence weight for each model")

# Plot model comparison
az.plot_compare(comparison, figsize=(8, 4))
plt.title("LOO Comparison: Seasonal Adjustment Methods", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Visualize the Seasonally Adjusted Counterfactual

In [ ]:
# Compare the visual output of naive vs month-dummy model

fig, axes = plt.subplots(2, 1, figsize=(13, 10))

# Naive model
_, ax_pair = model_naive.plot()
plt.close()  # Close the auto-generated figure

# Regenerate plots manually for side-by-side comparison
for ax, model, title in [
    (axes[0], model_naive, "Naive ITS (No Seasonal Adjustment)"),
    (axes[1], model_month_dummies, "ITS with Month Dummy Seasonal Controls"),
]:
    posterior = model.idata.posterior
    beta_lev = posterior["treated"].values.flatten().mean()
    ax.set_title(f"{title}\nEstimated level change: {beta_lev:+.1f} (true: {true_level_change:+.1f})", fontsize=12)

# Get proper plots
fig1, _ = model_naive.plot()
fig1.suptitle(f"Naive ITS — Estimated level change: {beta_naive.mean():+.1f} (true: {true_level_change:+.1f})",
              fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

fig2, _ = model_month_dummies.plot()
fig2.suptitle(f"Seasonally Adjusted ITS — Estimated level change: {beta_dummies.mean():+.1f} (true: {true_level_change:+.1f})",
              fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## Summary

### Key Lessons

1. **Seasonality confounds ITS when the intervention coincides with a seasonal transition.** In this example, the vaccination campaign launched in September — just before the winter flu season peak. Without seasonal adjustment, the natural seasonal rise in flu visits is attributed to the campaign.

2. **Month dummy variables** provide non-parametric seasonal adjustment. Each calendar month gets its own fixed effect. Requires at least 12 pre-intervention months (one complete cycle).

3. **Fourier terms** provide smooth parametric seasonal adjustment using fewer parameters. Better for shorter time series or when the seasonal pattern is smooth.

4. **LOO cross-validation** provides a principled comparison of seasonal adjustment methods. Use it when uncertain between approaches.

### When to Apply Seasonal Adjustment

- Always check the raw time series for seasonal patterns (visual inspection)
- If the intervention coincides with a seasonal transition, adjustment is essential
- If the pre-period spans less than 12 months, use Fourier terms (fewer parameters)
- If the seasonal pattern is complex (multi-peak), month dummies are more flexible

### What's Next
**Module 02** covers Bayesian ITS internals:
- Building ITS from scratch in PyMC
- Prior specification and sensitivity
- Posterior predictive checks for causal claims

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])